# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Display dataset title and description by accessing metadata attributes
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets and fields by `@id`
record_sets = dataset.metadata.recordSets
print("Available Record Sets:\n-------------------")
for rs in record_sets:
    print(f"@id: {rs['@id']} - name: {rs.get('name', 'N/A')}")
    # List fields in this record set
    if 'fields' in rs:
        print("  Fields:")
        for field in rs['fields']:
            fid = field['@id']
            fname = field.get('name', 'N/A')
            dtype = field.get('dataType', 'N/A')
            print(f"    @id: {fid} | name: {fname} | type: {dtype}")
    print()

# For example, print a sample of records from the first record set (if any)
if len(record_sets) > 0:
    example_rs_id = record_sets[0]['@id']
    print(f"\nExample records from record set with @id={example_rs_id}:")
    for i, rec in enumerate(dataset.records(record_set=example_rs_id)):
        print(rec)
        if i >= 2:
            break

## 3. Data Extraction
Load data from specific record set(s) into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
# We'll gather all record set `@id`s
record_sets = dataset.metadata.recordSets
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df

# Display fields/columns of the primary table (assume first record set)
if record_set_ids:
    main_id = record_set_ids[0]
    print(f"Columns in record set '@id': {main_id}")
    print(dataframes[main_id].columns.tolist())
    dataframes[main_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data. These operations help prepare for further analysis.

In [ ]:
# For EDA, automatically pick a numeric field if available
import numpy as np

# Helper: find first numeric field in the main DataFrame
main_df = dataframes[main_id]

numeric_field_candidates = main_df.select_dtypes(include=[np.number]).columns.tolist()
if numeric_field_candidates:
    numeric_field = numeric_field_candidates[0]
    print(f"Selected numeric field: {numeric_field}")
else:
    # Fallback: try to convert possible numeric fields
    for col in main_df.columns:
        try:
            main_df[col] = pd.to_numeric(main_df[col], errors='ignore')
        except Exception:
            pass
    numeric_field_candidates = main_df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_field_candidates:
        numeric_field = numeric_field_candidates[0]
        print(f"Selected numeric field after conversion: {numeric_field}")
    else:
        raise Exception("No numeric field found in the main record set.")

# Filtering: pick a reasonable threshold (e.g., above mean)
threshold = main_df[numeric_field].mean()
filtered_df = main_df[main_df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
print(filtered_df.head())

# Normalizing
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized '{numeric_field}' for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a likely categorical field (pick field with few unique values)
group_field = None
for col in main_df.columns:
    if col != numeric_field and main_df[col].dtype == object and main_df[col].nunique() > 1 and main_df[col].nunique() <= 5:
        group_field = col
        break

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nGrouped filtered data by '{group_field}':\n", grouped_df.head())
else:
    print("No suitable grouping field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the distribution of the numeric field
plt.figure(figsize=(8,4))
sns.histplot(main_df[numeric_field].dropna(), kde=True, bins=30)
plt.title(f"Distribution of '{numeric_field}'")
plt.xlabel(numeric_field)
plt.ylabel("Frequency")
plt.show()

# If grouping field found, show a boxplot
if group_field:
    plt.figure(figsize=(8,4))
    sns.boxplot(data=main_df, x=group_field, y=numeric_field)
    plt.title(f"{numeric_field} by {group_field}")
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.


* The FAIR² Croissant dataset contains mass spectrometry data from human mast cell extracellular vesicles, structured in clearly defined record sets and fields.
* This notebook demonstrated how to load and inspect the dataset using only the schema, referencing all entities by their `@id` per the Croissant specification.
* Basic EDA was performed: filtering, normalization, and grouping based on field properties present in the schema, leveraging all fields and columns by their `@id` when programmatically relevant.
* Using `mlcroissant`, similar steps can be adapted to other multi-table datasets adhering to Croissant schemas for reliable, reproducible processing workflows.
